In [2]:
!pip install duckdb --upgrade
import sys
sys.exit(0)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 104.2 MB/s  0:00:00m0:00:01
  Attempting uninstall: duckdb
    Found existing installation: duckdb 1.4.4
    Uninstalling duckdb-1.4.4:
sys.exit called with value 0. The interpreter will be restarted.
      Successfully uninstalled duckdb-1.4.4


_**you have to install it in a separate step**_

In [3]:
import duckdb
duckdb.sql(""" force install delta from core_nightly ; force install azure from core_nightly; update extensions """).show()
import sys
sys.exit(0)

┌────────────────┬──────────────┬─────────────────────┬──────────────────┬─────────────────┐
│ extension_name │  repository  │    update_result    │ previous_version │ current_version │
│    varchar     │   varchar    │       varchar       │     varchar      │     varchar     │
├────────────────┼──────────────┼─────────────────────┼──────────────────┼─────────────────┤
│ azure          │ core_nightly │ NO_UPDATE_AVAILABLE │ 563589b          │ 563589b         │
│ delta          │ core_nightly │ NO_UPDATE_AVAILABLE │ 29d43b9          │ 29d43b9         │
└────────────────┴──────────────┴─────────────────────┴──────────────────┴─────────────────┘

sys.exit called with value 0. The interpreter will be restarted.


In [4]:
table_path = 'abfss://1c52481c-0523-4a5a-bbde-fdc932bd77c2@onelake.dfs.fabric.microsoft.com/1cdb4ce3-c31f-44fa-9244-aeb76a4608db/Tables/dbo/ooxr'
source     = 'abfss://tpch@onelake.dfs.fabric.microsoft.com/storage.Lakehouse/Files/csv/PUBLIC_DAILY_201*.CSV'

In [5]:
import duckdb
con = duckdb.connect()
con.sql(F"""
    create or replace view source as 
    WITH raw AS (
        SELECT * FROM read_csv('{source}',skip=1, header=0, all_varchar=1,
            columns={{
                'I': 'VARCHAR','UNIT': 'VARCHAR','XX': 'VARCHAR','VERSION': 'VARCHAR','SETTLEMENTDATE': 'VARCHAR','RUNNO': 'VARCHAR',
                'DUID': 'VARCHAR','INTERVENTION': 'VARCHAR','DISPATCHMODE': 'VARCHAR','AGCSTATUS': 'VARCHAR','INITIALMW': 'VARCHAR',
                'TOTALCLEARED': 'VARCHAR','RAMPDOWNRATE': 'VARCHAR','RAMPUPRATE': 'VARCHAR','LOWER5MIN': 'VARCHAR',
                'LOWER60SEC': 'VARCHAR','LOWER6SEC': 'VARCHAR','RAISE5MIN': 'VARCHAR','RAISE60SEC': 'VARCHAR',
                'RAISE6SEC': 'VARCHAR','MARGINAL5MINVALUE': 'VARCHAR','MARGINAL60SECVALUE': 'VARCHAR',
                'MARGINAL6SECVALUE': 'VARCHAR','MARGINALVALUE': 'VARCHAR','VIOLATION5MINDEGREE': 'VARCHAR',
                'VIOLATION60SECDEGREE': 'VARCHAR','VIOLATION6SECDEGREE': 'VARCHAR','VIOLATIONDEGREE': 'VARCHAR',
                'LOWERREG': 'VARCHAR','RAISEREG': 'VARCHAR','AVAILABILITY': 'VARCHAR','RAISE6SECFLAGS': 'VARCHAR',
                'RAISE60SECFLAGS': 'VARCHAR','RAISE5MINFLAGS': 'VARCHAR','RAISEREGFLAGS': 'VARCHAR',
                'LOWER6SECFLAGS': 'VARCHAR','LOWER60SECFLAGS': 'VARCHAR','LOWER5MINFLAGS': 'VARCHAR',
                'LOWERREGFLAGS': 'VARCHAR','RAISEREGAVAILABILITY': 'VARCHAR','RAISEREGENABLEMENTMAX': 'VARCHAR',
                'RAISEREGENABLEMENTMIN': 'VARCHAR','LOWERREGAVAILABILITY': 'VARCHAR','LOWERREGENABLEMENTMAX': 'VARCHAR',
                'LOWERREGENABLEMENTMIN': 'VARCHAR','RAISE6SECACTUALAVAILABILITY': 'VARCHAR',
                'RAISE60SECACTUALAVAILABILITY': 'VARCHAR','RAISE5MINACTUALAVAILABILITY': 'VARCHAR',
                'RAISEREGACTUALAVAILABILITY': 'VARCHAR','LOWER6SECACTUALAVAILABILITY': 'VARCHAR',
                'LOWER60SECACTUALAVAILABILITY': 'VARCHAR','LOWER5MINACTUALAVAILABILITY': 'VARCHAR','LOWERREGACTUALAVAILABILITY': 'VARCHAR'
            }},
            filename=1, null_padding=true, ignore_errors=1, auto_detect=false
                       )
			WHERE I='D' AND UNIT='DUNIT' AND VERSION='3'
    )
    SELECT
        UNIT,
        DUID,
        COLUMNS(*EXCLUDE(DUID, UNIT, SETTLEMENTDATE,filename,I,XX))::DOUBLE,
        NULL AS week,
		filename as file,
		SETTLEMENTDATE::TIMESTAMPTZ AS SETTLEMENTDATE,
        SETTLEMENTDATE::DATE AS date,
        YEAR(SETTLEMENTDATE::TIMESTAMP) AS year
    FROM raw
""")

In [6]:
from deltalake import write_deltalake
arrow_data = con.sql(""" FROM source LIMIT 0 """).arrow()
write_deltalake(
    f"{table_path}",
    arrow_data,
    mode="ignore",
    #partition_by=["year"]
)

In [7]:
con.sql(f"""
               SET preserve_insertion_order = false;
               ATTACH OR REPLACE '{table_path}' as destination (type delta,read_only false ) ;
               INSERT INTO        destination FROM source  ;
               CHECKPOINT         destination
        """)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┐
│ Success │
│ boolean │
└─────────┘
  0 rows 